# Interactive Model Prediction Testing

**Project:** Active Medical Chatbot with ANN-based Diagnosis System

**Date:** 2025-10-18

---

## Overview

This notebook provides interactive tools to test the trained PyTorch ANN model with:

- **Test Set Samples:** Random samples from the test set
- **Custom Symptom Input:** Manual symptom combinations
- **Edge Cases:** Symptoms outside the training data
- **Top-K Predictions:** Multiple candidate diseases with confidence scores
- **Comparison:** Predicted vs actual diagnoses

---

## Objectives

1. Validate model predictions on known test cases
2. Test model behavior with custom symptom combinations
3. Explore model confidence and top-K accuracy
4. Identify cases requiring active questioning
5. Test edge cases and out-of-distribution inputs

## 1. Environment Setup

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 120)

print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Libraries imported successfully
PyTorch version: 2.9.0
Device: CPU


## 2. Configuration and Paths

In [2]:
# Define paths (adjust if running from different directory)
BASE_DIR = Path('..')  # Assumes notebook is in building_ANN/notebooks/
DATA_DIR = BASE_DIR / 'data' / 'processed' / 'common'
MODEL_DIR = BASE_DIR / 'models' / 'pytorch'

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print("INTERACTIVE PREDICTION TESTING")
print("="*80)
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Device: {DEVICE}")

INTERACTIVE PREDICTION TESTING
Data directory: ../data/processed/common
Model directory: ../models/pytorch
Device: cpu


## 3. Load Model and Data

In [3]:
# Load training data to get symptom names
print("Loading training data (for symptom names)...")
train_df = pd.read_csv(DATA_DIR / 'train.csv')
symptom_columns = [col for col in train_df.columns if col != 'disease']
num_symptoms = len(symptom_columns)

print(f"Found {num_symptoms} symptoms")
print(f"\nFirst 10 symptoms:")
for i, symptom in enumerate(symptom_columns[:10], 1):
    print(f"   {i}. {symptom}")

Loading training data (for symptom names)...
Found 377 symptoms

First 10 symptoms:
   1. lo lắng và căng thẳng
   2. trầm cảm
   3. hụt hơi
   4. các triệu chứng trầm cảm hoặc loạn thần
   5. đau ngực dữ dội
   6. chóng mặt
   7. mất ngủ
   8. các chuyển động không tự nguyện bất thường
   9. tức ngực
   10. hồi hộp


In [4]:
# Load test data
print("\nLoading test dataset...")
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f"Test dataset loaded")
print(f"   Test samples: {len(test_df):,}")
print(f"   Unique diseases: {test_df['disease'].nunique()}")


Loading test dataset...
Test dataset loaded
   Test samples: 28,327
   Unique diseases: 577


In [5]:
# Load label encoder
print("\nLoading label encoder...")
with open(MODEL_DIR / 'label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

num_classes = len(label_encoder.classes_)
print(f"Label encoder loaded")
print(f"   Number of diseases: {num_classes}")


Loading label encoder...
Label encoder loaded
   Number of diseases: 577


In [6]:
# Define model architecture (same as training)
class ANNDiseaseClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout_rates=(0.3, 0.3, 0.2)):
        super(ANNDiseaseClassifier, self).__init__()
        
        self.fc1 = nn.Linear(input_dim, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(dropout_rates[0])
        
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(dropout_rates[1])
        
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout3 = nn.Dropout(dropout_rates[2])
        
        self.fc4 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)
        
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout3(x)
        
        x = self.fc4(x)
        return x

print("Model architecture defined")

Model architecture defined


In [7]:
# Load trained model
print("Loading trained model...")
model = ANNDiseaseClassifier(num_symptoms, num_classes)
model = model.to(DEVICE)

# Load best model checkpoint
checkpoint = torch.load(MODEL_DIR / 'best_model.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded successfully")
print(f"   Best epoch: {checkpoint['epoch'] + 1}")
print(f"   Validation accuracy: {checkpoint['val_acc']*100:.2f}%")
print(f"   Validation loss: {checkpoint['val_loss']:.4f}")

Loading trained model...
Model loaded successfully
   Best epoch: 77
   Validation accuracy: 83.75%
   Validation loss: 0.4244


## 4. Helper Functions for Prediction

In [8]:
def predict_disease(symptom_vector, model, label_encoder, device, top_k=10):
    """
    Predict disease from symptom vector.
    
    Args:
        symptom_vector: numpy array of shape (num_symptoms,) with binary 0/1 values
        model: trained PyTorch model
        label_encoder: sklearn LabelEncoder for diseases
        device: torch device
        top_k: number of top predictions to return
    
    Returns:
        Dictionary with predictions and confidence scores
    """
    # Convert to tensor
    x = torch.FloatTensor(symptom_vector).unsqueeze(0).to(device)
    
    # Get prediction
    with torch.no_grad():
        output = model(x)
        probs = torch.softmax(output, dim=1)
        
        # Get top-k predictions
        top_probs, top_indices = torch.topk(probs, k=top_k, dim=1)
        
    # Convert to numpy
    top_probs = top_probs.cpu().numpy()[0]
    top_indices = top_indices.cpu().numpy()[0]
    
    # Get disease names
    top_diseases = label_encoder.inverse_transform(top_indices)
    
    return {
        'predictions': list(zip(top_diseases, top_probs)),
        'top_disease': top_diseases[0],
        'top_confidence': top_probs[0],
        'all_probs': probs.cpu().numpy()[0]
    }

print("Prediction function defined")

Prediction function defined


In [9]:
def display_prediction(result, actual_disease=None, show_symptoms=None):
    """
    Display prediction results in a formatted way.
    
    Args:
        result: prediction result dictionary
        actual_disease: actual disease name (if known)
        show_symptoms: list of symptom names that are present
    """
    print("="*80)
    print("PREDICTION RESULTS")
    print("="*80)
    
    if show_symptoms:
        print(f"\nINPUT SYMPTOMS ({len(show_symptoms)})")
        for i, symptom in enumerate(show_symptoms, 1):
            print(f"   {i}. {symptom}")
    
    print(f"\nTOP PREDICTION:")
    print(f"   Disease: {result['top_disease']}")
    print(f"   Confidence: {result['top_confidence']*100:.2f}%")
    
    if actual_disease:
        is_correct = result['top_disease'] == actual_disease
        status = "CORRECT" if is_correct else "INCORRECT"
        print(f"\n   Actual disease: {actual_disease}")
        print(f"   Status: {status}")
        
        # Check if actual disease is in top-k
        top_diseases = [pred[0] for pred in result['predictions']]
        if not is_correct and actual_disease in top_diseases:
            rank = top_diseases.index(actual_disease) + 1
            actual_conf = result['predictions'][rank-1][1]
            print(f"   Actual disease rank: #{rank} (confidence: {actual_conf*100:.2f}%)")
    
    print(f"\nTOP-10 PREDICTIONS:")
    for i, (disease, confidence) in enumerate(result['predictions'], 1):
        marker = "" if i == 1 else "  "
        if actual_disease and disease == actual_disease:
            marker = ""
        bar_length = int(confidence * 50)
        bar = "█" * bar_length
        print(f"   {marker} {i:2d}. {disease:50s} {confidence*100:6.2f}% {bar}")
    
    # Confidence category
    conf = result['top_confidence']
    if conf > 0.85:
        category = "HIGH (>85%) - Direct prediction recommended"
        icon = "🟢"
    elif conf > 0.60:
        category = "MEDIUM (60-85%) - Present top-3 options"
        icon = "🟡"
    else:
        category = "LOW (<60%) - Active questioning required"
        icon = "🔴"
    
    print(f"\n{icon} CONFIDENCE CATEGORY: {category}")
    print("="*80)

print("Display function defined")

Display function defined


In [10]:
def create_symptom_vector(symptom_names, all_symptom_columns):
    """
    Create a binary symptom vector from symptom names.
    
    Args:
        symptom_names: list of symptom names (Vietnamese)
        all_symptom_columns: list of all possible symptom columns
    
    Returns:
        Binary numpy array of shape (num_symptoms,)
    """
    vector = np.zeros(len(all_symptom_columns), dtype='float32')
    
    for symptom in symptom_names:
        if symptom in all_symptom_columns:
            idx = all_symptom_columns.index(symptom)
            vector[idx] = 1.0
        else:
            print(f"⚠️  Warning: Symptom '{symptom}' not found in training data")
    
    return vector

print("Symptom vector creation function defined")

Symptom vector creation function defined


## 5. Test with Random Samples from Test Set

In [11]:
# Test with a random sample from test set
print("Testing with random sample from test set...\n")

# Select random sample
random_idx = np.random.randint(0, len(test_df))
sample = test_df.iloc[random_idx]

# Extract symptoms and actual disease
actual_disease = sample['disease']
symptom_vector = sample[symptom_columns].values.astype('float32')
present_symptoms = [symptom_columns[i] for i in range(len(symptom_vector)) if symptom_vector[i] == 1]

# Predict
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)

# Display
display_prediction(result, actual_disease=actual_disease, show_symptoms=present_symptoms)

Testing with random sample from test set...

PREDICTION RESULTS

INPUT SYMPTOMS (7)
   1. tầm nhìn giảm sút
   2. các triệu chứng của mắt
   3. cảm giác có dị vật trong mắt
   4. đỏ mắt
   5. chảy nước mắt
   6. khối u trên mí mắt
   7. mắt sưng

TOP PREDICTION:
   Disease: chắp mắt
   Confidence: 100.00%

   Actual disease: chắp mắt
   Status: CORRECT

TOP-10 PREDICTIONS:
     1. chắp mắt                                           100.00% █████████████████████████████████████████████████
       2. lẹo mắt                                              0.00% 
       3. viêm bờ mi                                           0.00% 
       4. bệnh cườm nước mãn tính                              0.00% 
       5. trầy xước giác mạc                                   0.00% 
       6. nhiễm trùng giác mạc                                 0.00% 
       7. móng chân mọc ngược                                  0.00% 
       8. rối loạn tâm thần                                    0.00% 
       9. chèn ép

## 7. Custom Symptom Input - Example Cases

Now you can test with your own symptom combinations!

In [12]:
# Example 1: Common cold symptoms
print("\n TEST CASE 1: Common Cold Symptoms\n")

custom_symptoms = [
    'ho',
    'nghẹt mũi',
    'đau họng',
    'sốt',
    'Mệt mỏi',
    'đau đầu'
]

# Create symptom vector
symptom_vector = create_symptom_vector(custom_symptoms, symptom_columns)

# Predict
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)

# Display
display_prediction(result, show_symptoms=custom_symptoms)


 TEST CASE 1: Common Cold Symptoms

PREDICTION RESULTS

INPUT SYMPTOMS (6)
   1. ho
   2. nghẹt mũi
   3. đau họng
   4. sốt
   5. Mệt mỏi
   6. đau đầu

TOP PREDICTION:
   Disease: bệnh bạch cầu đơn nhân
   Confidence: 29.97%

TOP-10 PREDICTIONS:
     1. bệnh bạch cầu đơn nhân                              29.97% ██████████████
       2. bệnh phổi kẽ                                        28.83% ██████████████
       3. áp xe ở họng                                        22.92% ███████████
       4. viêm phế quản cấp tính                              12.04% ██████
       5. loét áp-tơ                                           3.28% █
       6. cảm lạnh thông thường                                1.28% 
       7. cúm                                                  0.69% 
       8. bệnh Lyme                                            0.68% 
       9. rối loạn mũi                                         0.24% 
      10. bệnh bạch cầu                                        0.03% 

🔴 CONF

In [ ]:
# Edge Case 4: Symptom not in training data
print("\n EDGE CASE 4: Symptom NOT in Training Data\n")

unknown_symptoms = [
    'sốt',
    'đau đầu',
    'triệu chứng không tồn tại',  # This symptom doesn't exist
    'ho'
]
symptom_vector = create_symptom_vector(unknown_symptoms, symptom_columns)
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
display_prediction(result, show_symptoms=[s for s in unknown_symptoms if s in symptom_columns])

print("\n Observation: Unknown symptoms are ignored (treated as 0).")
print("   The model only uses symptoms it was trained on.")


 EDGE CASE 4: Symptom NOT in Training Data

⚠️  Warning: Symptom 'triệu chứng không tồn tại' not found in training data
PREDICTION RESULTS

INPUT SYMPTOMS (3)
   1. sốt
   2. đau đầu
   3. ho

TOP PREDICTION:
   Disease: viêm màng não
   Confidence: 23.91%

TOP-10 PREDICTIONS:
     1. viêm màng não                                       23.91% ███████████
       2. bệnh bạch cầu đơn nhân                              20.64% ██████████
       3. loét áp-tơ                                          16.29% ████████
       4. áp xe ở họng                                        14.64% ███████
       5. bệnh phổi kẽ                                        11.30% █████
       6. cúm                                                  4.16% ██
       7. cảm lạnh thông thường                                2.81% █
       8. viêm họng liên cầu khuẩn                             1.37% 
       9. viêm phế quản cấp tính                               1.34% 
      10. bệnh Lyme                              

In [ ]:
# Edge Case 3: No symptoms (all zeros)
print("\n EDGE CASE 3: No Symptoms (All Zeros)\n")

zero_symptoms = []
symptom_vector = np.zeros(len(symptom_columns), dtype='float32')
result = predict_disease(symptom_vector, model, label_encoder, DEVICE, top_k=10)
display_prediction(result, show_symptoms=zero_symptoms)

print("\n Observation: With no symptoms, the model predicts based on class frequency.")
print("   Top predictions are likely the most common diseases in the training set.")


🧪 EDGE CASE 3: No Symptoms (All Zeros)

PREDICTION RESULTS

TOP PREDICTION:
   Disease: dị vật trong tai
   Confidence: 13.91%

TOP-10 PREDICTIONS:
     1. dị vật trong tai                                    13.91% ██████
       2. lão thính học                                        6.39% ███
       3. chứng khó nuốt                                       5.15% ██
       4. bàng quang mất trương lực                            3.54% █
       5. hẹp ống dẫn nước mắt                                 2.16% █
       6. rối loạn tuyến nước bọt                              1.97% 
       7. sốt thấp khớp mãn tính                               1.57% 
       8. chấn thương đầu gối                                  1.22% 
       9. ung thư đại trực tràng                               1.20% 
      10. đốt cháy                                             1.19% 

🔴 CONFIDENCE CATEGORY: LOW (<60%) - Active questioning required

💡 Observation: With no symptoms, the model predicts based on class frequen